In [ ]:
import brax.envs as brax_envs
import gymnax
from evosax.problems.bbob import bbob_fns

problems_brax_envs = list(brax_envs._envs.keys())
problems_bbob_fns = list(bbob_fns.keys())
problems_torch_vision = ["MNIST", "FashionMNIST", "CIFAR10", "SVHN"]
problems_gymnax = gymnax.registered_envs

problems = problems_brax_envs + problems_bbob_fns + problems_torch_vision + problems_gymnax

In [ ]:
import os
import json
import numpy as np
import plotly.graph_objects as go

def load_data(problem_dir):
    algorithms_data = {}
    
    # Iterate through each algorithm directory
    for algorithm in os.listdir(problem_dir):
        algorithm_path = os.path.join(problem_dir, algorithm)
        
        if os.path.isdir(algorithm_path):
            # Initialize list to hold data for each run
            algorithm_runs = []
            
            # Iterate through the JSON files
            for json_file in sorted(os.listdir(algorithm_path)):
                try:
                    json_path = os.path.join(algorithm_path, json_file)
                    
                    if json_file.endswith(".json"):
                        with open(json_path, 'r') as f:
                            data = json.load(f)
                            algorithm_runs.append(data)
                except:
                    pass
            algorithms_data[algorithm] = algorithm_runs
    
    return algorithms_data

def calculate_mean_std(algorithms_data):
    mean_std_data = {}
    
    for algorithm, runs in algorithms_data.items():
        # Convert each run's data to arrays
        best_fitness_arr = np.array([run['best_fitness'] for run in runs])
        
        # Calculate mean and std for each generation
        mean_best_fitness = np.mean(best_fitness_arr, axis=0)
        std_best_fitness = np.std(best_fitness_arr, axis=0)
        
        mean_std_data[algorithm] = {
            'mean': mean_best_fitness,
            'std': std_best_fitness
        }
    
    return mean_std_data

def plot_comparison(mean_std_data):
    fig = go.Figure()
    
    # Add traces for each algorithm
    for algorithm, data in mean_std_data.items():
        generations = np.arange(len(data['mean']))  # Generation count (x-axis)
        
        fig.add_trace(go.Scatter(
            x=generations, 
            y=data['mean'], 
            mode='lines', 
            name=f'{algorithm} Mean',
            line=dict(width=3)
        ))
        
        fig.add_trace(go.Scatter(
            x=generations, 
            y=data['mean'] + data['std'], 
            fill='tonexty', 
            mode='lines', 
            name=f'{algorithm} +1 Std',
            fillcolor='rgba(0, 100, 80, 0.2)',
            line=dict(width=0)
        ))
        
        fig.add_trace(go.Scatter(
            x=generations, 
            y=data['mean'] - data['std'], 
            fill='tonexty', 
            mode='lines', 
            name=f'{algorithm} -1 Std',
            fillcolor='rgba(0, 100, 80, 0.2)',
            line=dict(width=0)
        ))
    
    # Update layout
    fig.update_layout(
        title='Algorithm Comparison',
        xaxis_title='Generation',
        yaxis_title='Best Fitness',
        template='plotly_dark',
        showlegend=True
    )
    
    fig.show()

# Main workflow
problem_dir="../experiment_results/BBOBProblem/sphere"
algorithms_data = load_data(problem_dir)
mean_std_data = calculate_mean_std(algorithms_data)
plot_comparison(mean_std_data)

In [ ]:
from evosax.problems.bbob.bbob import BBOBProblem

BBOBProblem